# Week 2: Policy Improvement for the SPADE–MetaWorld Environment Designer

This notebook extends the Week 1 environment-designer loop with a **small, explicit MDP**. MetaWorld's continuous robot observations are deliberately not used as tabular states. Instead, a state describes the current curriculum-design stage, and an action describes the next SPADE edit.

The implementation persists the MDP, policies, algorithm runs, Bellman snapshots, state values, and action values in SQLite.

## 1. Minimal MDP

The MDP is $\mathcal{M}=(\mathcal{S},\mathcal{A},P,R,\gamma)$ with $\gamma=0.95$.

**States:** `invalid`, `foundation`, `varied`, `intermediate`, `target`, plus terminal states `accepted` and `rejected`. Nonterminal states correspond to complexity bands computed by Week 1's existing `complexity()` function.

**Actions:** `repair`, `increase_variation`, `advance_family`, `tighten_constraints`, and `accept`. Only actions with transition rows are available in a state.

**Reward:** every edit costs 1–2 points; premature acceptance receives a negative terminal reward; accepting a valid target environment receives +15. Transition probabilities are a transparent planning assumption, not measured MetaWorld success probabilities. A future version can estimate them from repeated Week 1 proposals and validation results.

In [ ]:
from sqlalchemy import select
from sqlalchemy.orm import Session

from spade_metaworld.policy_improvement import (
    DesignAction,
    DesignState,
    Transition,
    ValueSnapshot,
    create_policy_database,
    evaluate_policy_exactly,
    evaluate_policy_iteratively,
    named_policy,
    named_values,
    policy_iteration,
    seed_designer_mdp,
    snapshot_values,
)

engine = create_policy_database()
db = Session(engine)
world, uniform_policy = seed_designer_mdp(db)
print(f'World: {world.name}; gamma={world.discount}')

## 2. SQL-backed data model

`policy_mdp.sql` is the authoritative schema. SQLAlchemy classes map those tables rather than defining a second domain model. Important tables are:

- `design_state`, `design_action`, `transition`: the MDP itself;
- `policy`, `policy_probability`: $\pi(a\mid s)$;
- `algorithm_run`, `value_snapshot`: reproducible experiment history;
- `state_value`, `action_value`: $V(s)$ and one-step $Q(s,a)$ for every saved snapshot.

In [ ]:
states = list(db.scalars(select(DesignState).order_by(DesignState.id)))
for state in states:
    band = (state.complexity_min, state.complexity_max)
    print(f'{state.name:13s} terminal={bool(state.is_terminal)!s:5s} complexity={band}')

In [ ]:
actions = {a.id: a.name for a in db.scalars(select(DesignAction))}
state_names = {s.id: s.name for s in states}
for transition in db.scalars(select(Transition).order_by(Transition.state_id, Transition.action_id)):
    print(
        f'{state_names[transition.state_id]:13s} --{actions[transition.action_id]:20s}--> '
        f'{state_names[transition.next_state_id]:13s} '
        f'p={transition.probability:.2f} r={transition.reward:+.1f}'
    )

## 3. Policy evaluation

For a fixed policy, synchronous evaluation applies the Bellman expectation backup

$$V_{k+1}(s)=\sum_a\pi(a\mid s)\sum_{s'}P(s'\mid s,a)[R(s,a,s')+\gamma V_k(s')].$$

The stopping condition is $\max_s|V_{k+1}(s)-V_k(s)|\leq 10^{-10}$. As an independent check, exact evaluation solves

$$(I-\gamma P_\pi)V_\pi=r_\pi.$$

In [ ]:
iterative_run, iterative_snapshot = evaluate_policy_iteratively(db, world, uniform_policy)
_, exact_snapshot = evaluate_policy_exactly(db, world, uniform_policy)
iterative_values = snapshot_values(db, iterative_snapshot)
exact_values = snapshot_values(db, exact_snapshot)

print(f'Converged after {iterative_run.iterations} sweeps')
print('state          iterative       exact        absolute error')
for state in states:
    error = abs(iterative_values[state.id] - exact_values[state.id])
    print(f'{state.name:13s} {iterative_values[state.id]:12.6f} {exact_values[state.id]:12.6f} {error:14.3e}')

In [ ]:
import matplotlib.pyplot as plt

history = db.execute(
    select(ValueSnapshot.sequence, ValueSnapshot.bellman_residual)
    .where(ValueSnapshot.run_id == iterative_run.id)
    .order_by(ValueSnapshot.sequence)
).all()
plt.figure(figsize=(7, 4))
plt.plot([row.sequence + 1 for row in history], [row.bellman_residual for row in history])
plt.yscale('log')
plt.xlabel('Policy-evaluation sweep')
plt.ylabel('Maximum Bellman update')
plt.title('Uniform-policy evaluation convergence')
plt.grid(alpha=0.25)
plt.show()

## 4. Greedy policy improvement and policy iteration

For every available action, one-step lookahead computes

$$Q_\pi(s,a)=\sum_{s'}P(s'\mid s,a)[R(s,a,s')+\gamma V_\pi(s')].$$

The improved policy assigns equal probability to all actions tied within numerical tolerance for $\max_a Q_\pi(s,a)$. Policy iteration alternates exact evaluation and this greedy update until no state's action probabilities change.

In [ ]:
iteration_run, optimal_policy, optimal_snapshot = policy_iteration(db, world, uniform_policy)
optimal_values = named_values(db, world, snapshot_values(db, optimal_snapshot))
optimal_actions = named_policy(db, world, optimal_policy)

print(f'Policy stable after {iteration_run.iterations} evaluation/improvement rounds')
for state_name, action_names in optimal_actions.items():
    print(f'{state_name:13s} V*={optimal_values[state_name]:8.4f}  actions={action_names}')

## 5. Interpretation and limitations

The resulting policy repairs invalid proposals, advances or tightens valid environments, and accepts only the target state. This is a policy over **environment-design operations**, not a robot-control policy.

The probabilities in this minimal assignment are declared assumptions so that dynamic programming has a complete model. The natural next experiment is to execute each editing action many times through the existing `RuleBasedDesigner` and `EnvironmentValidator`, classify the result with `classify_environment()`, and replace the assumed probabilities with observed transition frequencies.

In [ ]:
db.commit()
db.close()